# 02 — Data Pattern Analysis

**Research question:** before cleaning any text field, what's actually in it? Special characters like `&`, `feat.`, parentheses, and dashes show up constantly in artist and song names — some are real collaboration markers worth stripping, some are just part of the name. This stage measures what's there; it doesn't decide what to clean.

**Machine counts, human judges.** Unlike 01 (every check is machine-concluded), most steps here only get you halfway: the engine classifies a pattern's *type*, but whether that type should actually be removed is a call only a human can make after reading the report. Step 6 is the one step that isn't engine output at all — it's the hand-written judgment call, informed by everything above it.

## Architecture

| | |
|---|---|
| **CONFIG** | `PATTERNS_ARTIST` / `PATTERNS_SONG` (what to detect, shared across all 3 datasets) + `DATASETS` (the only thing that actually differs per dataset: column names) |
| **Engine** | 4 generic functions — `run_pattern_scan()`, `classify_symbol_usage()`, `classify_extracted_content()` (+ its `classify_parenthetical_content()` wrapper), `show_category_examples()` — none of them know they're looking at Billboard vs. Spotify |
| **Steps 1-5** | Feed D1/D2/D3 through the engine, one report per step — reports state facts, they don't recommend action |
| **Step 6** | `CLEANING_RULES` — the one deliverable that's hand-authored, not engine output: a judgment call made after reading every report above |

**Scope note:** R's original version only ran the Step 2-4 deep classification on D1 — D2/D3 got a written conclusion with no classification code behind it. I deliberately expanded Steps 2-4 to run on all three datasets here, since the classification functions were already generic and D2/D3 turned out to have the same kind of pattern issues D1 did — worth measuring instead of guessing.

## Step Blueprint

| Step | What It Checks | Question Answered | Who Concludes |
|------|------|------|------|
| **Step 1** | Basic Pattern Statistics (`run_pattern_scan`) | How often does each special character (`&`, `feat.`, parentheses...) show up, and what % of rows? | Machine (pure counting) |
| **Step 2** | `&` / `,` Classification (`classify_symbol_usage`) | Does this symbol sit next to a collaboration marker (featuring/feat./with)? Does it appear more than once? | Machine classifies → **human reads the report to decide** |
| **Step 3** | Parenthetical Content Classification (`classify_extracted_content` + `PAREN_CATEGORIES`) | Is what's in the parentheses a version tag (Remix/Live/Remaster), or part of the song/artist name itself? | Machine classifies → **human judgment** |
| **Step 4** | Post-Dash Content Classification (same function, a different `DASH_CATEGORIES`) | Is what follows a dash a version tag, or a song subtitle? | Machine classifies → **human judgment** |
| **Step 5** | Cross-Dataset Comparison (`three_way` table) | D1 vs. D2 vs. D3 — how much does the same pattern's prevalence differ? Which dataset is "dirtiest"? | Machine computes → **human reads the trend** |
| **Step 6** | `CLEANING_RULES` (final deliverable) | Given everything above, which patterns should actually be cleaned, which should stay? | **Pure human judgment** — the only thing here that isn't engine output |

In [1]:
import pandas as pd

# Load validated datasets from Stage 01
d1_billboard   = pd.read_pickle('../Data/cleaned_D1.pkl')
d2_spotify_all = pd.read_pickle('../Data/cleaned_D2.pkl')
d3_music       = pd.read_pickle('../Data/cleaned_D3.pkl')

print('Datasets loaded')
print(f'D1 (Billboard): {len(d1_billboard):,} rows')
print(f'D2 (Spotify):   {len(d2_spotify_all):,} rows')
print(f'D3 (Music):     {len(d3_music):,} rows')

Datasets loaded
D1 (Billboard): 330,087 rows
D2 (Spotify):   41,106 rows
D3 (Music):     28,372 rows


## CONFIG — Only edit this section when adding datasets or patterns

`PATTERNS_ARTIST` / `PATTERNS_SONG` are written once, since the patterns worth detecting (`&`, `feat.`, parentheses...) are the same across D1/D2/D3. The only thing that actually differs per dataset is the **column name** (D1: `artist`/`song`, D2: `artist`/`track`, D3: `artist_name`/`track_name`), so that lives separately in `DATASETS`.

Adding a pattern to detect later (e.g. non-Latin characters) means editing `PATTERNS_ARTIST` in one place — all three datasets pick it up automatically.

In [2]:
# ============================================================
# CONFIG -- Only edit this section when adding datasets/patterns 
# ============================================================

# case=False below means matching is case-insensitive -- write a pattern in any
# case, all variants get caught automatically.
# Patterns shared across ALL datasets' artist/artist_name column
PATTERNS_ARTIST = [
    ("Contains 'Featuring'",       r'Featuring'),
    ("Contains 'feat.' or 'ft.'",  r'feat\.|ft\.'),
    ("Contains 'with' (Collab)",   r'\bwith\b'),   # Step 2's COLLAB_MARKER_REGEX already treats 'with'
                                                     # as a collab marker -- counted here too for consistency
    ("Contains '&'",               r'&'),
    ("Contains ','",               r','),
    ("Contains '()' or '[]'",      r'[\(\)\[\]]'), # ( or ) or [ or ]
    ("Contains '/'",               r'/'),
    ("Contains 'x' (Collab)",      r'\sx\s'),      # 'x' surrounded by spaces on both sides
    ("Contains '+' (Collab)",      r'\s\+\s'),     # '+' surrounded by spaces on both sides
]

# Patterns shared across ALL datasets' song/track/track_name column
PATTERNS_SONG = [
    ("Contains '()'",           r'\(.*\)'),       # capture full content wrapped in ( )
    ("Contains '[]'",           r'\[.*\]'),       # capture full content wrapped in [ ]
    ("Contains ' - ' (dash)",   r' - '),
    ("Contains 'feat.'",        r'feat\.'),
    ("Contains 'Remaster'",     r'Remaster'),
    ("Contains 'Live'",         r'Live'),
    ("Contains 'Remix'",        r'Remix'),
    ("Contains 'Radio Edit'",   r'Radio Edit'),
]

# Column-name mapping per dataset -- this is the ONLY thing that
# actually differs between D1 / D2 / D3
DATASETS = {
    "D1": {"df": d1_billboard,   "artist_col": "artist",      "song_col": "song"},
    "D2": {"df": d2_spotify_all, "artist_col": "artist",      "song_col": "track"},
    "D3": {"df": d3_music,       "artist_col": "artist_name", "song_col": "track_name"},
}

## Engine — No edits needed below this line

`run_pattern_scan()` takes a dataframe, a column, and a pattern list, and returns a stats table (count + percentage).

**What it deliberately doesn't do:** decide what should be cleaned. It only answers "how much" — the judgment call happens after a human reads the report.

In [3]:
def run_pattern_scan(df, col, patterns):
    """Scan one column against a list of (name, regex) patterns.
    Returns a stats DataFrame -- does NOT decide what to clean."""

    total = len(df)
    rows = []
    for pattern_name, regex in patterns:
        count = df[col].str.contains(regex, case=False, na=False).sum()
        pct = round(count / total * 100, 2)
        rows.append({"Pattern": pattern_name, "Count": count, "Pct": pct})

    return pd.DataFrame(rows)


def show_examples(df, col, regex, n=3):
    """Print n example values matching a pattern -- for human judgment."""
    examples = (
        df[df[col].str.contains(regex, case=False, na=False)][col]
        .drop_duplicates()
        .head(n)
        .tolist()
    )
    for ex in examples:
        print(f"    -> {ex}")

## Step 1 — Basic Pattern Statistics

One loop covers D1/D2/D3's artist column and song column — 6 reports total, instead of writing the same scan three times.

In [4]:
print("[STEP 1] Basic Pattern Statistics")

for dataset_name, cfg in DATASETS.items():
    df = cfg["df"]
    artist_col = cfg["artist_col"]
    song_col = cfg["song_col"]

    print(f"\n{'='*55}")
    print(f"  {dataset_name} -- artist column: '{artist_col}'")
    print(f"{'='*55}")
    artist_stats = run_pattern_scan(df, artist_col, PATTERNS_ARTIST)
    print(artist_stats.to_string(index=False))

    print(f"\n{'='*55}")
    print(f"  {dataset_name} -- song column: '{song_col}'")
    print(f"{'='*55}")
    song_stats = run_pattern_scan(df, song_col, PATTERNS_SONG)
    print(song_stats.to_string(index=False))

[STEP 1] Basic Pattern Statistics

  D1 -- artist column: 'artist'


                  Pattern  Count  Pct
     Contains 'Featuring'  32274 9.78
Contains 'feat.' or 'ft.'    441 0.13
 Contains 'with' (Collab)   3535 1.07
             Contains '&'  25155 7.62
             Contains ','   4206 1.27
    Contains '()' or '[]'   1324 0.40
             Contains '/'    766 0.23
    Contains 'x' (Collab)    844 0.26
    Contains '+' (Collab)    651 0.20

  D1 -- song column: 'song'


              Pattern  Count  Pct
        Contains '()'  22882 6.93
        Contains '[]'    112 0.03
Contains ' - ' (dash)    872 0.26
     Contains 'feat.'      0 0.00
  Contains 'Remaster'      0 0.00
      Contains 'Live'   1770 0.54
     Contains 'Remix'     94 0.03
Contains 'Radio Edit'      0 0.00

  D2 -- artist column: 'artist'
                  Pattern  Count  Pct
     Contains 'Featuring'   1658 4.03
Contains 'feat.' or 'ft.'     10 0.02
 Contains 'with' (Collab)    128 0.31
             Contains '&'   1786 4.34
             Contains ','    255 0.62
    Contains '()' or '[]'     63 0.15
             Contains '/'     40 0.10
    Contains 'x' (Collab)     40 0.10
    Contains '+' (Collab)     24 0.06

  D2 -- song column: 'track'
              Pattern  Count  Pct
        Contains '()'   2831 6.89
        Contains '[]'     92 0.22
Contains ' - ' (dash)   2857 6.95
     Contains 'feat.'    148 0.36
  Contains 'Remaster'    865 2.10
      Contains 'Live'    604 1.47
     Contains

                  Pattern  Count  Pct
     Contains 'Featuring'      0 0.00
Contains 'feat.' or 'ft.'      2 0.01
 Contains 'with' (Collab)      9 0.03
             Contains '&'    844 2.97
             Contains ','    224 0.79
    Contains '()' or '[]'      3 0.01
             Contains '/'     45 0.16
    Contains 'x' (Collab)      9 0.03
    Contains '+' (Collab)     16 0.06

  D3 -- song column: 'track_name'
              Pattern  Count  Pct
        Contains '()'   1480 5.22
        Contains '[]'     24 0.08
Contains ' - ' (dash)      0 0.00
     Contains 'feat.'    357 1.26
  Contains 'Remaster'     24 0.08
      Contains 'Live'    234 0.82
     Contains 'Remix'     28 0.10
Contains 'Radio Edit'      5 0.02


## CONFIG — Step 2-4 Classification Rules

R's original version only ran this deeper layer of analysis on D1; D2/D3 got a hand-written conclusion with no classification code behind it. Ported faithfully here — D1 gets full classification, D2/D3 initially inherited R's manual conclusions (later verified with real classification evidence in Steps 2-4 below).

These CONFIG blocks answer a more specific question than Step 1's pattern lists: not just "does this symbol exist," but "what does it actually mean" (a version tag? a collab credit? just part of the title?).

In [5]:
# ============================================================
# CONFIG -- classification rules for Step 2-4 (Deep Analysis)
# ============================================================

# CONFIG -- classification rules for symbols that need collab-marker context
COLLAB_MARKER_REGEX = r'featuring|feat\.|ft\.|with'

# CONFIG -- classification rules for parenthetical content (song titles)
SONG_PAREN_CATEGORIES = [
    ("Remix Version",         r'remix'),
    ("Performance Version",   r'^live$|live version|acoustic|unplugged'),
    ("Release Version Tag",   r'radio edit|album version|single version'),
    ("Remastered Version",    r'remaster'),
    ("Featuring Info",        r'feat\.|featuring'),
]
SONG_PAREN_DEFAULT = "Subtitle or Title Part (Keep)"

# CONFIG -- artist column uses a DIFFERENT, simpler rule (mirrors R exactly --
# do NOT reuse SONG_PAREN_CATEGORIES here, they answer a different question)
ARTIST_PAREN_CATEGORIES = [
    ("Collab-related (Safe to remove)", r'feat|duet|featuring'),
]
ARTIST_PAREN_DEFAULT = "Group Info/Metadata (Preserve)"

# CONFIG -- content after a " - " dash (song titles only, D1)
DASH_CATEGORIES = [
    ("Remix Version",         r'remix'),
    ("Remastered Version",    r'remaster'),
    ("Performance Version",   r'live|acoustic'),
    ("Release Version Tag",   r'radio edit|album version|single'),
]
DASH_DEFAULT = "Subtitle or Title Part (Keep)"

## Engine — Step 2-4 Classification Functions

`classify_symbol_usage()` and `classify_extracted_content()` / `classify_parenthetical_content()` are fully generic — every classification rule lives in the CONFIG above, so applying this to a new dataset later is a matter of calling the function, not rewriting logic.

`show_category_examples()` prints up to 5 real examples per category, so a reader isn't left guessing what a category actually looks like from a percentage alone. Steps 2/3/4 all call the same function rather than each writing their own.

In [6]:
def classify_symbol_usage(df, col, symbol_regex):
    """Classify rows containing `symbol_regex` by collab-marker context + repeat count.
    Mirrors R's Analysis 1 (& symbol) / Analysis 3 (comma) case_when() logic."""
    data = df[df[col].str.contains(symbol_regex, na=False)][[col]].drop_duplicates().copy()
    data["has_collab_marker"] = data[col].str.contains(COLLAB_MARKER_REGEX, case=False, na=False)
    data["symbol_count"] = data[col].str.count(symbol_regex)

    def _type(row):
        if row["has_collab_marker"]:
            return "Has collab marker (Safe to clean)"
        elif row["symbol_count"] >= 2:
            return "Multiple (Complex)"
        else:
            return "Simple (Likely duo/group or part of name)"

    data["type"] = data.apply(_type, axis=1)
    total = len(data)
    summary = (data["type"].value_counts()
               .rename_axis("Type").reset_index(name="Count"))
    summary["Pct"] = round(summary["Count"] / total * 100, 2)
    return data, summary


def classify_extracted_content(df, col, contains_regex, extract_regex, categories, default_label):
    """Filter rows matching `contains_regex`, extract the substring matched by
    `extract_regex` (group 1), then classify it by `categories` (first match wins,
    else `default_label`). Generalizes R's Analysis-2-style case_when() blocks --
    used for both parentheses content and post-dash content."""
    mask = df[col].str.contains(contains_regex, na=False)
    data = df[mask][[col]].drop_duplicates().copy()
    data["extracted"] = data[col].str.extract(extract_regex)[0]

    def _type(text):
        if pd.isna(text):
            return default_label
        for label, regex in categories:
            if pd.Series([text]).str.contains(regex, case=False, regex=True).iloc[0]:
                return label
        return default_label

    data["type"] = data["extracted"].apply(_type)
    total = len(data)
    summary = (data["type"].value_counts()
               .rename_axis("Type").reset_index(name="Count"))
    summary["Pct"] = round(summary["Count"] / total * 100, 2)
    return data, summary


def classify_parenthetical_content(df, col, categories, default_label):
    """Thin wrapper: parentheses is the most common case of classify_extracted_content."""
    # thin wrapper around classify_extracted_content()
    return classify_extracted_content(
        df, col, r'\(.*\)', r'\(([^)]+)\)', categories, default_label)


def show_category_examples(data, col, n=5):
    """Print up to n example values per category -- for human eyeballing.

    `data` must be the classified DataFrame returned by classify_symbol_usage()
    or classify_extracted_content() (has a 'type' column alongside the original
    text column). Shows real examples per category so a reader isn't guessing
    what a category looks like from a percentage alone."""
    for type_label, group in data.groupby("type"):
        examples = group[col].head(n).tolist()
        print(f"    [{type_label}]  ({len(group):,} total, showing up to {n})")
        for ex in examples:
            print(f"      -> {ex}")

### Step 2 — `&` / `,` Classification (D1 / D2 / D3)

**Scope expanded beyond R's original version:** R only classified D1 here; D2/D3 got a conclusion written from Step 1's percentages alone. Since `classify_symbol_usage()` was already dataset-agnostic, this runs on all three — real evidence instead of an educated guess about what a symbol means in D2/D3.

In [7]:
print("[STEP 2] '&' / ',' Usage Classification (D1 / D2 / D3)")

step2_results = {}
for dataset_name, cfg in DATASETS.items():
    df = cfg["df"]
    artist_col = cfg["artist_col"]

    print(f"\n{'='*55}")
    print(f"  {dataset_name} -- artist column: '{artist_col}'")
    print(f"{'='*55}")

    print(f"\n=== {dataset_name} {artist_col}: '&' usage classification ===")
    amp_data, amp_summary = classify_symbol_usage(df, artist_col, "&")
    print(f"Total containing '&': {len(amp_data):,}\n")
    print(amp_summary.to_string(index=False))
    print()
    show_category_examples(amp_data, artist_col, n=5)

    print(f"\n=== {dataset_name} {artist_col}: ',' usage classification ===")
    comma_data, comma_summary = classify_symbol_usage(df, artist_col, ",")
    print(f"Total containing ',': {len(comma_data):,}\n")
    print(comma_summary.to_string(index=False))
    print()
    show_category_examples(comma_data, artist_col, n=5)

    step2_results[dataset_name] = {
        "amp_data": amp_data, "amp_summary": amp_summary,
        "comma_data": comma_data, "comma_summary": comma_summary,
    }

[STEP 2] '&' / ',' Usage Classification (D1 / D2 / D3)

  D1 -- artist column: 'artist'

=== D1 artist: '&' usage classification ===
Total containing '&': 1,485

                                     Type  Count   Pct
Simple (Likely duo/group or part of name)    866 58.32
        Has collab marker (Safe to clean)    612 41.21
                       Multiple (Complex)      7  0.47

    [Has collab marker (Safe to clean)]  (612 total, showing up to 5)
      -> Drake Featuring Future & Young Thug
      -> Wizkid Featuring Justin Bieber & Tems
      -> Nardo Wick Featuring G Herbo, Lil Durk & 21 Savage
      -> Drake Featuring 21 Savage & Project Pat
      -> Bryson Gray Featuring Tyson James & Chandler Crump
    [Multiple (Complex)]  (7 total, showing up to 5)
      -> Bad Bunny, Jowell & Randy & Nengo Flow
      -> Luke Combs & Brooks & Dunn
      -> Rihanna & Kanye West & Paul McCartney
      -> John Travolta & Olivia Newton-John & Cast
      -> Delaney & Bonnie & Friends
    [Simple (Li

    [Has collab marker (Safe to clean)]  (379 total, showing up to 5)
      -> Cheech & Chong Featuring Alice Bowie
      -> Pratt & McClain with Brother Love
      -> Lisa Lisa And Cult Jam With Full Force Featuring Paul Anthony & Bow Legged Lou
      -> Tom Petty & The Heartbreakers With Stevie Nicks
      -> Kenny Rogers With Kim Carnes & James Ingram
    [Multiple (Complex)]  (4 total, showing up to 5)
      -> Delaney & Bonnie & Friends
      -> Ike & Tina Turner & The Ikettes
      -> Rihanna & Kanye West & Paul McCartney
      -> Luke Combs & Brooks & Dunn
    [Simple (Likely duo/group or part of name)]  (538 total, showing up to 5)
      -> Shirley & Lee
      -> Ike & Tina Turner
      -> Jr. Walker & The All Stars
      -> Don Hinson & The Rigamorticians
      -> Diana Ross & The Supremes

=== D2 artist: ',' usage classification ===
Total containing ',': 171

                                     Type  Count   Pct
        Has collab marker (Safe to clean)     89 52.05
Simple (

### Step 3 — Parenthetical Content Classification (D1 / D2 / D3)

Artist and song columns use **different CONFIGs** (`ARTIST_PAREN_CATEGORIES` vs. `SONG_PAREN_CATEGORIES`) — same function, different rules, because parentheses mean different things in each: a song's parens are usually a version tag, an artist's parens are usually a collab credit.

**Methodology note:** this classifies unique artist/song strings, not raw chart rows (`.drop_duplicates()` runs before classifying) — kept consistent with Step 2. R's own original code was *not* internally consistent here (it deduplicated for the `&` analysis but not for this one) — confirmed this is an inherent inconsistency in R's original code, not a translation error, and standardized on the unique-string basis throughout.

In [8]:
print("[STEP 3] Parenthetical Content Classification (D1 / D2 / D3)")

step3_results = {}
for dataset_name, cfg in DATASETS.items():
    df = cfg["df"]
    artist_col = cfg["artist_col"]
    song_col = cfg["song_col"]

    print(f"\n{'='*55}")
    print(f"  {dataset_name}")
    print(f"{'='*55}")

    print(f"\n=== {dataset_name} {artist_col}: parentheses content classification ===")
    artist_paren_data, artist_paren_summary = classify_parenthetical_content(
        df, artist_col, ARTIST_PAREN_CATEGORIES, ARTIST_PAREN_DEFAULT)
    print(f"Total containing parentheses: {len(artist_paren_data):,}\n")
    print(artist_paren_summary.to_string(index=False))
    print()
    show_category_examples(artist_paren_data, artist_col, n=5)

    print(f"\n=== {dataset_name} {song_col}: parentheses content classification ===")
    song_paren_data, song_paren_summary = classify_parenthetical_content(
        df, song_col, SONG_PAREN_CATEGORIES, SONG_PAREN_DEFAULT)
    print(f"Total containing parentheses: {len(song_paren_data):,}\n")
    print(song_paren_summary.to_string(index=False))
    print()
    show_category_examples(song_paren_data, song_col, n=5)

    step3_results[dataset_name] = {
        "artist_paren_data": artist_paren_data, "artist_paren_summary": artist_paren_summary,
        "song_paren_data": song_paren_data, "song_paren_summary": song_paren_summary,
    }

[STEP 3] Parenthetical Content Classification (D1 / D2 / D3)

  D1

=== D1 artist: parentheses content classification ===


Total containing parentheses: 87

                           Type  Count   Pct
Collab-related (Safe to remove)     45 51.72
 Group Info/Metadata (Preserve)     42 48.28

    [Collab-related (Safe to remove)]  (45 total, showing up to 5)
      -> R. Kelly Or Bow Wow (Featuring T.I. & T-Pain)
      -> Puff Daddy & The Family (Feat. The Notorious B.I.G. & Mase)
      -> Anita Cochran (Duet With Steve Wariner)
      -> Changing Faces (Featuring Jay-Z)
      -> SWV (Featuring Puff Daddy)
    [Group Info/Metadata (Preserve)]  (42 total, showing up to 5)
      -> Silk Sonic (Bruno Mars & Anderson .Paak)
      -> F.L.Y. (Fast Life Yungstaz)
      -> The Swell Season (Glen Hansard & Marketa Irglova)
      -> (+44)
      -> M.V.P. (Most Valuable Playas) Featuring Stagga Lee

=== D1 song: parentheses content classification ===


Total containing parentheses: 1,995

                         Type  Count   Pct
Subtitle or Title Part (Keep)   1983 99.40
          Performance Version      6  0.30
                Remix Version      5  0.25
               Featuring Info      1  0.05

    [Featuring Info]  (1 total, showing up to 5)
      -> Black Lassie (Featuring Johnny Stash)
    [Performance Version]  (6 total, showing up to 5)
      -> As Long As You Love Me (Acoustic)
      -> The Prayer (Live)
      -> Freak On A Leash (Unplugged)
      -> I'll Be Home For Christmas (Live)
      -> I Will Remember You (Live)
    [Remix Version]  (5 total, showing up to 5)
      -> Cold Heart (PNAU Remix)
      -> No New Friends (SFTB Remix)
      -> Karate Chop (Remix)
      -> Outta Control (Remix)
      -> Sympathy For The Devil (Remixes)
    [Subtitle or Title Part (Keep)]  (1,983 total, showing up to 5)
      -> Montero (Call Me By Your Name)
      -> Love Nwantiti (Ah Ah Ah)
      -> Get Into It (Yuh)
      -> Ya Superame 

Total containing parentheses: 2,736

                         Type  Count   Pct
Subtitle or Title Part (Keep)   2434 88.96
               Featuring Info    132  4.82
           Remastered Version     97  3.55
          Performance Version     40  1.46
                Remix Version     27  0.99
          Release Version Tag      6  0.22

    [Featuring Info]  (132 total, showing up to 5)
      -> In Love Wit Chu (feat. Cherish) - Radio Edit
      -> Lonesome River (feat. Ricky Skaggs & Keith Whitley)
      -> Rock Bottom (feat. Ricky Skaggs & Keith Whitley)
      -> Canción de Cuna para Despertar un Hijo (feat. Marilina Ross)
      -> Shouting on the Hills of Glory (feat. Ricky Skaggs & Keith Whitley)
    [Performance Version]  (40 total, showing up to 5)
      -> Levee Camp Blues (Live)
      -> Rock And Roll All Nite (live)
      -> Remember I Was Vapour (Live)
      -> The Lovin' blues (Live)
      -> Kaun Kahta Hai (Live)
    [Release Version Tag]  (6 total, showing up to 5)
      -

Total containing parentheses: 1,409

                         Type  Count   Pct
Subtitle or Title Part (Keep)   1007 71.47
               Featuring Info    335 23.78
          Performance Version     25  1.77
                Remix Version     20  1.42
           Remastered Version     15  1.06
          Release Version Tag      7  0.50

    [Featuring Info]  (335 total, showing up to 5)
      -> don't look back (feat. van morrison)
      -> whenever i call you "friend" (feat. stevie nicks)
      -> i just can't stop loving you (feat. siedah garrett)
      -> gin and juice (feat. dat nigga daz)
      -> whatta man (feat. en vogue)
    [Performance Version]  (25 total, showing up to 5)
      -> so far away (live)
      -> i feel the earth move (live)
      -> two occasions (live)
      -> be ok (acoustic)
      -> nothing i hold on to (live)
    [Release Version Tag]  (7 total, showing up to 5)
      -> total trash (album version)
      -> black night (single version)
      -> alabama bl

### Step 4 — Post-Dash Content Classification (D1 / D2 / D3)

D3's `track_name` already tested at 0% dash usage in Step 1 — the loop below runs anyway and skips automatically when there's nothing to classify.

In [9]:
print("[STEP 4] Post-Dash Content Classification (D1 / D2 / D3)")

step4_results = {}
for dataset_name, cfg in DATASETS.items():
    df = cfg["df"]
    song_col = cfg["song_col"]

    print(f"\n{'='*55}")
    print(f"  {dataset_name} -- {song_col}")
    print(f"{'='*55}")

    dash_mask_count = df[song_col].str.contains(r' - ', na=False).sum()
    if dash_mask_count == 0:
        print(f"  No ' - ' occurrences, skipping.")
        continue

    dash_data, dash_summary = classify_extracted_content(
        df, song_col, r' - ', r' - (.*)$', DASH_CATEGORIES, DASH_DEFAULT)
    print(f"Total containing ' - ': {len(dash_data):,}\n")
    print(dash_summary.to_string(index=False))
    print()
    show_category_examples(dash_data, song_col, n=5)

    step4_results[dataset_name] = {"dash_data": dash_data, "dash_summary": dash_summary}

[STEP 4] Post-Dash Content Classification (D1 / D2 / D3)

  D1 -- song
Total containing ' - ': 96

                         Type  Count   Pct
Subtitle or Title Part (Keep)     95 98.96
                Remix Version      1  1.04

    [Remix Version]  (1 total, showing up to 5)
      -> Roxanne `97 - Puff Daddy Remix
    [Subtitle or Title Part (Keep)]  (95 total, showing up to 5)
      -> Savage Love (Laxed - Siren Beat)
      -> Never Leave You - Uh Ooh, Uh Oooh!
      -> Undone - The Sweater Song
      -> Don't Look Down - The Sequel
      -> Dance Wit' Me - Part 1

  D2 -- track


Total containing ' - ': 2,758

                         Type  Count   Pct
Subtitle or Title Part (Keep)   1439 52.18
           Remastered Version    764 27.70
          Performance Version    377 13.67
                Remix Version    113  4.10
          Release Version Tag     65  2.36

    [Performance Version]  (377 total, showing up to 5)
      -> In My Father's House - Live
      -> Yambu - Live
      -> Western générique - Version Live
      -> Sailor Man - Live
      -> Space - Live At The Jazz Workshop, Boston/1967
    [Release Version Tag]  (65 total, showing up to 5)
      -> In Love Wit Chu (feat. Cherish) - Radio Edit
      -> The Peking Theme (So Little Time) - Single Version
      -> Hanky Panky - Single Version
      -> Weary Blues From Waitin' - Single Version / Dubbed
      -> I Am The Fly - Single Version
    [Remastered Version]  (764 total, showing up to 5)
      -> Carolina - Remastered 2006
      -> Alfômega - Remastered 2006
      -> Comment te dire adieu - Rema

## Step 5 — Cross-Dataset Comparison

D1 vs. D2 vs. D3 — artist column patterns, side by side.

In [10]:
print("[STEP 5] Cross-Dataset Comparison")

three_way = None
for dataset_name, cfg in DATASETS.items():
    stats = run_pattern_scan(cfg["df"], cfg["artist_col"], PATTERNS_ARTIST)
    stats = stats.rename(columns={"Pct": f"{dataset_name} %"})[["Pattern", f"{dataset_name} %"]]
    three_way = stats if three_way is None else three_way.merge(stats, on="Pattern")

print("\nD1 vs D2 vs D3 -- Artist Pattern Comparison (%)")
print("=" * 60)
print(three_way.to_string(index=False))

[STEP 5] Cross-Dataset Comparison



D1 vs D2 vs D3 -- Artist Pattern Comparison (%)
                  Pattern  D1 %  D2 %  D3 %
     Contains 'Featuring'  9.78  4.03  0.00
Contains 'feat.' or 'ft.'  0.13  0.02  0.01
 Contains 'with' (Collab)  1.07  0.31  0.03
             Contains '&'  7.62  4.34  2.97
             Contains ','  1.27  0.62  0.79
    Contains '()' or '[]'  0.40  0.15  0.01
             Contains '/'  0.23  0.10  0.16
    Contains 'x' (Collab)  0.26  0.10  0.03
    Contains '+' (Collab)  0.20  0.06  0.06


## Step 6 — `CLEANING_RULES` (final deliverable, hand-authored)

**Not engine output** — this translates every report from Steps 1-5, plus the classification evidence actually run against D1/D2/D3, into a structured config. Stage 03's `clean_column(series, rules)` reads this dict directly, so cleaning logic never needs a dataset-specific branch.

Both column types use one fixed schema regardless of dataset — "nothing to clean" is an empty list, never a different shape:
- **Artist-type columns** (D1 artist / D2 artist / D3 artist_name): `remove_after_marker` / `remove_parens_if_contains` / `preserve` / `note`
- **Song-type columns** (D1 song / D2 track / D3 track_name): `remove_version_tags_in_parens` / `remove_version_tags_after_dash` / `remove_brackets_entirely` / `preserve` / `note`

In [11]:
print("[STEP 6] CLEANING_RULES (final deliverable, unified schema)\n")

# Artist-type columns all use: remove_after_marker / remove_parens_if_contains / preserve / note
# Song-type columns all use:   remove_version_tags_in_parens / remove_version_tags_after_dash /
#                               remove_brackets_entirely / preserve / note
# Every dataset gets the SAME keys -- "nothing to clean" is an empty list, never a different shape.

CLEANING_RULES = {
    "D1": {
        "artist": {
            "remove_after_marker": [r"featuring", r"feat\.", r"ft\.", r"with"],
            "remove_parens_if_contains": [r"feat", r"duet", r"featuring"],
            "preserve": ["&", ",", "brackets not matching collab markers"],
            "note": "41% of '&' and 52% of ',' cases have a collab marker -- real cleanup needed, not just cosmetic",
        },
        "song": {
            # 2026/07/27 corrected against R's actual clean_d1_song() (03_data_wrangling.qmd lines 75-89).
            # Previous version here was a generalized guess, not a literal match -- it over-stripped
            # bare "live" after a dash (R never does this for D1) and under-stripped acoustic/unplugged/
            # featuring/ft. in parens (R does remove these for D1). See references/pipeline-history.md.
            "remove_version_tags_in_parens": [r"remix", r"remaster", r"live version", r"acoustic",
                                               r"unplugged", r"featuring", r"feat\.", r"ft\."],
            "remove_version_tags_after_dash": [r"remix"],  # R ONLY strips remix after a dash for D1
            "remove_brackets_entirely": True,
            "preserve": ["standalone parens/dash treated as title subtitle",
                         "'live'/'part'/'take' as lyric words -- false positive risk, do not blanket-remove"],
            "note": "Corrected 2026/07/27 to match R's clean_d1_song() exactly (see note above)",
        },
    },
    "D2": {
        "artist": {
            "remove_after_marker": [r"featuring", r"feat\.", r"ft\.", r"with"],
            "remove_parens_if_contains": [r"feat", r"duet", r"featuring"],
            "preserve": ["&", "/", "+", "x", "commas"],
            "note": "2026/07/20 evidence: 41% of '&' and 52% of ',' cases have a collab marker, "
                    "59% of parens are collab-related -- essentially the same profile as D1, "
                    "previously missing remove_parens_if_contains (D2 parens were wrongly left as blanket-preserve)",
        },
        "track": {
            # 2026/07/27 corrected against R's actual clean_d2_track() (03_data_wrangling.qmd lines
            # 163-182). R only removes "(live version|acoustic|unplugged)" from parens, and only
            # removes a dash clause that is EXACTLY "live"/"version live"/"live version" (anchored to
            # end of string) -- the old bare "live" tag matched ANY parens/dash content containing
            # "live" (e.g. "(Live)", "- Live @ Wacken"), which R does not touch. R also strips
            # "(featuring|feat.|ft.)" from parens -- old config only had "feat.". See references/pipeline-history.md.
            "remove_version_tags_in_parens": [r"remix", r"remaster", r"live version", r"acoustic",
                                               r"unplugged", r"featuring", r"feat\.", r"ft\.",
                                               r"radio edit"],
            "remove_version_tags_after_dash": [r"remaster", r"remix", r"radio edit", r"feat\."],
            # bare "live" deliberately dropped -- R only removes it as an EXACT dash-clause match,
            # which this engine's loose "contains" matching cannot safely replicate.
            "remove_brackets_entirely": True,
            "preserve": ["'Part'/'Pt.' after dash -- track numbering, not a version tag"],
            "note": "Corrected 2026/07/27 to match R's clean_d2_track() exactly (see note above)",
        },
    },
    "D3": {
        "artist_name": {
            "remove_after_marker": [],
            "remove_parens_if_contains": [],
            "preserve": ["&", ",", "essentially clean already"],
            "note": "2026/07/20 evidence: 0% Featuring, 100% of '&' cases are Simple (no collab marker), "
                    "only 1 row has parentheses at all (Group Info, preserve) -- no removal rules needed, "
                    "still gets the universal trim/normalize step in 03 like every other column",
        },
        "track_name": {
            "remove_version_tags_in_parens": [r"remix", r"remaster", r"feat\.",
                                               r"live version", r"acoustic", r"unplugged",
                                               r"radio edit", r"album version", r"single version"],
            "remove_version_tags_after_dash": [],
            "remove_brackets_entirely": True,
            "preserve": ["'live' outside parens is frequently a lyric word ('as long as i live') -- "
                          "only the precise phrases above (e.g. 'live version', not bare 'live') get removed"],
            "note": "2026/07/20 evidence: Featuring 23.8%, Performance Version 1.77%, Release Version Tag 0.50% "
                    "all showed up in real classification -- added those two categories that were previously "
                    "missing from the removal list. 0% dash usage, so remove_version_tags_after_dash is empty.",
        },
    },
}

print("CLEANING_RULES defined for D1 / D2 / D3 -- ready for 03_data_wrangling")
for ds, cols in CLEANING_RULES.items():
    print(f"  {ds}: {list(cols.keys())}")

[STEP 6] CLEANING_RULES (final deliverable, unified schema)

CLEANING_RULES defined for D1 / D2 / D3 -- ready for 03_data_wrangling
  D1: ['artist', 'song']
  D2: ['artist', 'track']
  D3: ['artist_name', 'track_name']
